# Open-weights rung - reliability probe on Colab (vLLM + Qwen3.6-27B)

Adds an **open-weights, white-box** model to the R90 ladder. Downloads **Qwen/Qwen3.6-27B** (dense, Apache-2.0, native **262,144**-token window - verified on the HF card 2026-06-22), serves it through vLLM's **OpenAI-compatible** endpoint (the probe's OpenAI adapter then works with only an env-var swap), and runs a **near-only smoke test** as the hard go/no-go before any GPU-expensive sweep.

**Footguns this notebook handles for you (all card-verified):**
- **VRAM:** 27B in BF16 is ~54 GB of weights - needs an **80 GB A100**. It will NOT fit a 40 GB A100. Cell 2 checks.
- **No YaRN:** the 262k native window already exceeds every fill we use, so `--rope-scaling` is left **OFF** (the card warns static YaRN degrades shorter texts - which would corrupt the near gate).
- **Thinking OFF:** Qwen3.6 thinks by default and has no `/no_think` switch; we force `enable_thinking=False` per request so it matches the non-reasoning API panel (gpt-3.5 / gpt-4o-mini / Haiku / Sonnet) and the tool call is not starved by a think block.
- **Tool parser:** `--tool-call-parser qwen3_coder` (the probe uses native function-calling).


In [ ]:
# --- config: confirm/swap the model here ---
MODEL = 'Qwen/Qwen3.6-27B'   # dense, Apache-2.0, native 262,144 window (HF card, 2026-06-22)
MODEL_DIR = '/content/model'
SERVED = 'open-model'        # id the probe passes as --model; must match --served-model-name
MAX_LEN = 131072             # WITHIN the 262,144 native window -> NO YaRN needed (see serve cell)
TOP_FILL = 96000             # cap so ctx (~1.3x fill) stays under MAX_LEN; raise both together
REPO_URL = 'https://github.com/R1ch1k/agent-reliability.git'


In [ ]:
# --- 0. confirm the hardware (decides feasibility) ---
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
print()
print('NEED ~80 GB: Qwen3.6-27B in BF16 is ~54 GB of weights; it does NOT fit a 40 GB A100.')
print('If memory.total is ~40 GB: Runtime > Change runtime type > A100 (80 GB), or fall back to a')
print('smaller CURRENT dense model at FULL precision. Do NOT 4-bit a 27B - quantization is the')
print('confound this white-box rung exists to avoid (near>=0.97 also guards against a broken quant).')


In [ ]:
# --- 1. install (vLLM bundles its CUDA/torch deps; slow on first run) ---
!pip install -q 'vllm>=0.19.0' huggingface_hub requests
import vllm; print('vLLM', vllm.__version__, '(need >=0.19.0 for the Qwen3.6 architecture)')


In [ ]:
# --- 2. download weights (point local_dir at a mounted Drive path to persist across sessions) ---
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=MODEL_DIR)   # Apache-2.0, ungated (no token)
print('downloaded to', MODEL_DIR)


In [ ]:
# --- 3. serve the OpenAI-compatible endpoint in the BACKGROUND, then poll /health ---
import subprocess, time, requests
# NATIVE window = 262,144 and MAX_LEN is below it, so DO NOT enable YaRN/--rope-scaling:
# the card warns static YaRN degrades shorter texts, which would corrupt the near gate.
args = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_DIR,
    '--served-model-name', SERVED,
    '--max-model-len', str(MAX_LEN),
    '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder',  # Qwen3.6 tool parser (card)
    '--enable-prefix-caching',          # reuse the shared manual prefix within a run
    '--gpu-memory-utilization', '0.92',
    '--port', '8000',
]
logf = open('/content/vllm.log', 'w')
srv = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for _ in range(360):                    # up to 30 min: 27B download-to-GPU load is slow
    try:
        if requests.get('http://localhost:8000/health').ok:
            ok = True; print('server UP'); break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print('server did NOT come up - last log lines:')
    print(''.join(open('/content/vllm.log').readlines()[-40:]))
    print('If you see a KV-cache / max_model_len error: lower MAX_LEN (e.g. 98304 or 65536) and TOP_FILL, then re-run from cell 1.')


In [ ]:
# --- 4. point the OpenAI SDK at the local server + force NON-thinking mode ---
import os, json
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'EMPTY'   # vLLM needs no auth
# Qwen3.6 THINKS BY DEFAULT and has no /no_think switch. The API panel was all non-reasoning,
# so for a capability-controlled comparison we force thinking OFF. The probe reads this env var
# (PROBE_OPENAI_EXTRA_BODY) and passes it as extra_body on every call.
os.environ['PROBE_OPENAI_EXTRA_BODY'] = json.dumps({'chat_template_kwargs': {'enable_thinking': False}})

from openai import OpenAI
r = OpenAI().chat.completions.create(
    model=SERVED,
    messages=[{'role': 'user', 'content': 'reply with the single word OK'}],
    max_tokens=16,
    extra_body=json.loads(os.environ['PROBE_OPENAI_EXTRA_BODY']),
)
print(repr(r.choices[0].message.content))   # expect 'OK' with NO think block


In [ ]:
# --- 5. clone the harness + run the NEAR-ONLY smoke test (the hard go/no-go) ---
!git clone -q $REPO_URL /content/probe
%cd /content/probe
# OPENAI_BASE_URL / OPENAI_API_KEY / PROBE_OPENAI_EXTRA_BODY from cell 4 propagate into this subprocess.
# --calib 0: this near sweep IS the gate, so skip the redundant built-in calib gate.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near --fills 8000 32000 64000 $TOP_FILL --runs 5 --needles 3 --depth 0.5 --calib 0


## Go / no-go

- **near holds ~1.0 (>=0.97) out to the top fill** -> capability gate passes; green-light the full sweep.
- **near craters at 64k-96k** -> usable range is short, or the serve config (quant/parser) is degrading it. Inspect `/content/vllm.log`; if tool calls are not parsing, the `--tool-call-parser` is the first suspect. Cap the sweep below the break, or fix serving, before spending GPU hours.

Then run the full curve (cell 9) and the length-disentangling pass (cell 10).

**Record for the write-up (the serving confound - the harness does NOT log these):** exact model tag, **quant = BF16**, **KV-cache dtype** (vLLM default = auto unless `--kv-cache-dtype` is set), `--max-model-len`, and that **YaRN was OFF** (native window). Expect a **mid-ladder** result (gpt-4o-mini / Haiku band), not Sonnet - open-model effective context sits well below advertised. That is the point of the white-box rung.


In [ ]:
# --- 6. FULL near+distance sweep (the white-box rung). Only run if near held above. ---
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills 8000 16000 32000 64000 $TOP_FILL --runs 15 --needles 3 --depth 0.5


In [ ]:
# --- 7. DISENTANGLING pass (review #4): isolate raw context-LENGTH from search-space size. ---
# --padding inert = FIXED 60-rule pool + inert filler (length grows, candidate count does not);
# --fixed-needle-seed = the SAME needles at every fill. Records now carry padding/needle_seed, so
# this curve is separable from the default (distractor-padded) one above even under the same --model.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills 8000 16000 32000 64000 $TOP_FILL --runs 15 --needles 3 --depth 0.5 --padding inert --fixed-needle-seed 7


## Save the data + add the rung to the ladder

Colab is ephemeral - **download the results** (cell 12) before the runtime recycles.

Back on your machine, to place Qwen3.6-27B on the R90 ladder:
1. Copy the **distractor-padded** `dist_results_*.jsonl` into the repo and add its filename to `canonical_manifest.txt`.
2. Add `'open-model'` (or your `--model` label) to the `order` list in `analyze_curves.py`, then re-run `python analyze_curves.py` and `python make_figures.py`.
3. The **inert-padded** file stays OUT of `canonical_manifest.txt` - it is the disentangling comparison (filter records by `padding == 'inert'`), not part of the headline ladder.


In [ ]:
# --- 8. zip + download all results (Colab is ephemeral) ---
from google.colab import files
import glob, zipfile
out = '/content/probe_results.zip'
with zipfile.ZipFile(out, 'w') as z:
    for f in glob.glob('/content/probe/dist_results_*.jsonl'):
        z.write(f, arcname=f.split('/')[-1])
    z.write('/content/vllm.log', arcname='vllm.log')
files.download(out)
print('zipped + downloading', out)
